In [30]:
import pandas as pd
from sklearn.model_selection import train_test_split
import os

from utils import load_data

In [31]:
data_path = '../data/cancer_reg-1.csv'
df = load_data(data_path)

### Drop PctSomeCol18_24 column since it contains missing values of more than 70% of the dataset


In [32]:
df = df.drop(columns=['PctSomeCol18_24'])

### Impute PctPrivateCoverageAlone(20% missing values) and PctEmployed16_Over(5% missing values) with the median

In [33]:
df['PctPrivateCoverageAlone'] = df['PctPrivateCoverageAlone'].fillna(df['PctPrivateCoverageAlone'].median())
df['PctEmployed16_Over'] = df['PctEmployed16_Over'].fillna(df['PctEmployed16_Over'].median())

In [34]:
df.isna().sum().sort_values(ascending=False)

avgAnnCount                0
PctBachDeg18_24            0
PctMarriedHouseholds       0
PctOtherRace               0
PctAsian                   0
PctBlack                   0
PctWhite                   0
PctPublicCoverageAlone     0
PctPublicCoverage          0
PctEmpPrivCoverage         0
PctPrivateCoverageAlone    0
PctPrivateCoverage         0
PctUnemployed16_Over       0
PctEmployed16_Over         0
PctBachDeg25_Over          0
PctHS25_Over               0
PctHS18_24                 0
avgDeathsPerYear           0
PctNoHS18_24               0
PercentMarried             0
AvgHouseholdSize           0
Geography                  0
MedianAgeFemale            0
MedianAgeMale              0
MedianAge                  0
binnedInc                  0
studyPerCap                0
povertyPercent             0
popEst2015                 0
medIncome                  0
incidenceRate              0
TARGET_deathRate           0
BirthRate                  0
dtype: int64

Now the numerial features don't have missing values anymore

### Handle categorical values

In [35]:
df[['binnedInc', 'Geography']]

,binnedInc,Geography
0,"(61494.5, 125635]","Kitsap County, Washington"
1,"(48021.6, 51046.4]","Kittitas County, Washington"
2,"(48021.6, 51046.4]","Klickitat County, Washington"
3,"(42724.4, 45201]","Lewis County, Washington"
4,"(48021.6, 51046.4]","Lincoln County, Washington"
...,...,...
3042,"(45201, 48021.6]","Ellsworth County, Kansas"
3043,"(48021.6, 51046.4]","Finney County, Kansas"
3044,"(51046.4, 54545.6]","Ford County, Kansas"
3045,"(48021.6, 51046.4]","Franklin County, Kansas"


For `binnedInc`, we transforms it into the mean values since it can be valuable rather than just the `medIncome` (median income)

In [36]:
def compute_mean(bin_income):
    """Compute the mean of a string like '(5000, 20000]'"""
    bin_income = bin_income.strip('()[]')
    lower, upper = bin_income.split(',')
    return (float(lower) + float(upper)) / 2

df['meanIncome'] = df['binnedInc'].apply(compute_mean)
df = df.drop(columns=['binnedInc'], axis=1)

In [37]:
df[['Geography']].describe()

,Geography
count,3047
unique,3047
top,"Kitsap County, Washington"
freq,1


For `Geography`, since all values are unique, we can safely remove this feature since encoding it won't provide valuable information

In [38]:
df = df.drop(columns=['Geography'], axis=1)

In [39]:
df

,avgAnnCount,avgDeathsPerYear,TARGET_deathRate,incidenceRate,medIncome,popEst2015,povertyPercent,studyPerCap,MedianAge,MedianAgeMale,...,PctEmpPrivCoverage,PctPublicCoverage,PctPublicCoverageAlone,PctWhite,PctBlack,PctAsian,PctOtherRace,PctMarriedHouseholds,BirthRate,meanIncome
0,1397.000000,469,164.9,489.800000,61898,260131,11.2,499.748204,39.3,36.9,...,41.6,32.9,14.0,81.780529,2.594728,4.821857,1.843479,52.856076,6.118831,93564.75
1,173.000000,70,161.3,411.600000,48127,43269,18.6,23.111234,33.0,32.2,...,43.6,31.1,15.3,89.228509,0.969102,2.246233,3.741352,45.372500,4.333096,49534.00
2,102.000000,50,174.7,349.700000,49348,21026,14.6,47.560164,45.0,44.0,...,34.9,42.1,21.1,90.922190,0.739673,0.465898,2.747358,54.444868,3.729488,49534.00
3,427.000000,202,194.8,430.400000,44243,75882,17.1,342.637253,42.8,42.2,...,35.0,45.3,25.0,91.744686,0.782626,1.161359,1.362643,51.021514,4.603841,43962.70
4,57.000000,26,144.4,350.100000,49955,10321,12.5,0.000000,48.3,47.8,...,35.1,44.0,22.7,94.104024,0.270192,0.665830,0.492135,54.027460,6.796657,49534.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3042,1962.667684,15,149.6,453.549422,46961,6343,12.4,0.000000,44.2,41.1,...,44.6,31.7,13.2,90.280811,3.837754,0.327613,1.700468,51.063830,7.773512,46611.30
3043,1962.667684,43,150.1,453.549422,48609,37118,18.8,377.175494,30.4,29.3,...,48.6,28.8,17.7,75.706245,2.326771,4.044920,14.130288,52.007937,8.186470,49534.00
3044,1962.667684,46,153.9,453.549422,51144,34536,15.0,1968.959926,30.9,30.5,...,47.8,26.6,16.8,87.961629,2.313188,1.316472,5.680705,55.153949,7.809192,52796.00
3045,1962.667684,52,175.0,453.549422,50745,25609,13.3,0.000000,39.0,36.9,...,49.6,29.5,14.0,92.905681,1.176562,0.244632,2.131790,58.484232,7.582938,49534.00


In [40]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3047 entries, 0 to 3046
Data columns (total 32 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   avgAnnCount              3047 non-null   float64
 1   avgDeathsPerYear         3047 non-null   int64  
 2   TARGET_deathRate         3047 non-null   float64
 3   incidenceRate            3047 non-null   float64
 4   medIncome                3047 non-null   int64  
 5   popEst2015               3047 non-null   int64  
 6   povertyPercent           3047 non-null   float64
 7   studyPerCap              3047 non-null   float64
 8   MedianAge                3047 non-null   float64
 9   MedianAgeMale            3047 non-null   float64
 10  MedianAgeFemale          3047 non-null   float64
 11  AvgHouseholdSize         3047 non-null   float64
 12  PercentMarried           3047 non-null   float64
 13  PctNoHS18_24             3047 non-null   float64
 14  PctHS18_24              

Now, all the features are numerical and don't contain any missing values, we can split our dataset for training

In [41]:
train_test_ratio = 0.7
val_test_ratio = 0.5
seed = 215

X = df.drop(columns=['TARGET_deathRate'], axis=1)
y = df['TARGET_deathRate']

X_train, X_valtest, y_train, y_valtest = train_test_split(X, y, train_size=train_test_ratio, random_state=seed)
X_val, X_test, y_val, y_test = train_test_split(X_valtest, y_valtest, test_size=val_test_ratio, random_state=seed)

### Shape of splitted data

In [42]:
print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_val: {X_val.shape}, y_val: {y_val.shape}")
print(f"X_test: {X_test.shape}, y_test: {y_test.shape}")

X_train: (2132, 31), y_train: (2132,)
X_val: (457, 31), y_val: (457,)
X_test: (458, 31), y_test: (458,)


### Save data for later training and evaluation

In [43]:
save_path = '../data/processed/'
os.makedirs(save_path, exist_ok=True)

X_train.to_csv(f'{save_path}X_train.csv', index=False)
X_val.to_csv(f'{save_path}X_val.csv', index=False)
X_test.to_csv(f'{save_path}X_test.csv', index=False)

y_train.to_csv(f'{save_path}y_train.csv', index=False)
y_val.to_csv(f'{save_path}y_val.csv', index=False)
y_test.to_csv(f'{save_path}y_test.csv', index=False)